In [ ]:
import os
import torch
import pandas as pd
import glob
import torchvision
from torchvision.io import read_image
from torch.utils.data import Dataset,random_split
import matplotlib.pyplot as plt
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.linear_model import LogisticRegression
import torch.nn as nn
import torch.nn.functional as  F
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
mounted_drive_path = '/content/drive/MyDrive/datasets'
os.listdir(mounted_drive_path)


In [ ]:
if 'dermoscopy_classification.tar.gz' in os.listdir(mounted_drive_path):
    # Copy the file to the current directory
    !cp '/content/drive/MyDrive/datasets/dermoscopy_classification.tar.gz' .

    # Extract the contents of the tar.gz file
    !tar -xvzf dermoscopy_classification.tar.gz

In [ ]:
data_dir = '/content/dermoscopy_classification'
data_dir

In [ ]:
path=glob.glob(data_dir + '/*/*.jpg') #some testing to get the path
path

In [ ]:
metadata_path=os.path.join(data_dir,'metadata.csv') #'/content/dermoscopy_classification/metadata.csv
metadata=pd.read_csv(metadata_path)
metadata

In [ ]:
from pathlib import Path

#Class MLProject2Dataset

In [ ]:
class MLProject2Dataset(Dataset):
  def __init__(self, data_dir, metadata_fname='metadata.csv', transform=None):
    #super().__init__
    self.data_dir=data_dir
    self.transform=transform

    metadata_path=os.path.join(data_dir,metadata_fname)
    metadata=pd.read_csv(metadata_path)



    path=glob.glob(data_dir + '/*/*.jpg')#take path for every image
    image_ids=[]
    for p in path:

      path_img=Path(p) #convert every path to a Path object
      image_id=path_img.name.replace('.jpg','')#take image_id of every Path object(example: ISIC_0029193.jpg)
      image_ids.append(image_id)


    self.df=pd.DataFrame({'image_id':image_ids,'path':path})


    metadata['dx']=pd.Categorical(metadata['dx']).codes #convert categorical variables to numbers
    self.df1=pd.merge(metadata,self.df, on='image_id') #inner join in order to find the matching id on metadata with my df
    self.data=self.df1[['image_id', 'path', 'dx']] #make the final df

  def __len__(self):
   return len(self.data)

  def __getitem__(self,idx):
    img_info=self.data.loc[idx]

    img_path=self.data.loc[idx]['path'] #take path from the given idx
    image=read_image(img_path)
    img=image.to(dtype=torch.float)/255
    if self.transform: #if transform != none
      img=self.transform(img)
    X=img #store image on X
    y=self.data.loc[idx]['dx'] #store label from the given idx on y

    return X,y

#change image size according to the query.I didn't find something better , so i change the size of image manually
m=50
#m=100
n=62
#n=125
transform = torchvision.transforms.Compose(
    [torchvision.transforms.Resize((m,n)),
     torchvision.transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
)



In [ ]:
#test to see if I have correctly fixed the dataset part
full_dataset=MLProject2Dataset(data_dir,transform=transform)
plt.imshow(full_dataset[0][0].permute(1,2,0))

In [ ]:
full_dataset.data #test to see if i have correctly fixed the data dataframe

#DataLoaders and train/val/test splits

In [ ]:
#split on  train val test set
train_size = int(0.6 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(42))


In [ ]:
len(train_dataset), len(val_dataset), len(test_dataset)

In [ ]:
#DataLoaders
batch_size=64
trainloader=DataLoader(train_dataset,batch_size=batch_size, shuffle=True)
valloader=DataLoader(val_dataset,batch_size=batch_size)
testloader=DataLoader(test_dataset,batch_size=batch_size)

In [ ]:
num_batches=len(trainloader)
num_batches

#Functions train_net and test_net

In [ ]:
def train_net(model: nn.Module, trainloader: DataLoader, valloader: DataLoader = None,
epochs: int = 10, optimizer: optim = None, loss: nn.modules.loss = None,
device: str = 'cpu', print_period: int = 10) -> None:
    model.train()
    train_losses = []
    train_accuracies = []
    val_losses=[]
    val_accuracies=[]


    for epoch in range(epochs):
      total_train=0
      val_loss=0
      correct=0

      running_loss = 0.0
      for batch, (X, y) in enumerate(trainloader,0):

          # Move to device
          X, y = X.to(device), y.to(device)
          y=y.type(torch.cuda.LongTensor) #need to cast to this type for gpu (find here :https://stackoverflow.com/questions/60440292/runtimeerror-expected-scalar-type-long-but-found-float)
          #y=y.type(torch.LongTensor) #for cpu

          # Zero gradients
          optimizer.zero_grad()


          # Forward pass
          pred = model(X)
          current_loss = loss(pred, y)


          # Backpropagation
          current_loss.backward()

          # Parameter update
          optimizer.step()

          # Print
          running_loss += current_loss.item()
          total_train += y.size(0)

          correct += (pred.argmax(1) == y).type(torch.float).sum().item()
          #print every 10 batches
          if batch % print_period == print_period-1:

              avg_loss = running_loss / print_period
              train_losses.append(avg_loss)
              acc=correct/total_train
              train_accuracies.append(acc)

              print(f'[Epoch:{epoch}, batch: {batch:5d}/{len(trainloader)}] '
                    f'Loss : {avg_loss:.3f}, Accuracy: {acc:.2f}')

              running_loss = 0.0
              total_train=0.0
              correct=0

          #val_loss and val_acc
          if valloader is not None and batch % print_period == print_period-1:
            model.eval()
            #use the test_net function with the valloader
            val_loss, val_acc,all_labels, all_preds = test_net(model, valloader, loss, device)
            print(f'[Epoch:{epoch}, batch: {batch:5d}] '
               f'Loss_val: {val_loss:.3f}, Accuracy_val: {val_acc:.2f}')

            val_accuracies.append(val_acc)
            val_losses.append(val_loss)

    # Plotting training loss
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Period')
    plt.ylabel('Loss')
    #plt.title('Training Loss')
    plt.legend()
    plt.show()

    # Plotting validation loss
    plt.figure(figsize=(8, 4))
    plt.plot(train_accuracies, label="Accuracy on training")
    plt.plot(val_accuracies, label="Val accuracy")
    plt.xlabel('Period')
    plt.ylabel('Accuracy')
    #plt.title('Accuracy')
    plt.legend()
    plt.show()


#function test_net
def test_net(model,testloader,loss,device):
    metric_loss=0.0
    total=0
    correct=0
    with torch.no_grad():
      for batch, (X,y) in enumerate(testloader, 0):
        X,y=X.to(device), y.to(device)
        y=y.type(torch.cuda.LongTensor)
        #y=y.type(torch.LongTensor)

        pred=model(X)
        current_loss=loss(pred,y)

        metric_loss+=current_loss.item()
        total +=y.size(0)
        correct += (pred.argmax(1) == y).type(torch.float).sum().item()
      avg_loss=metric_loss/len(testloader)
      #take actual and pred labels for the cnfusion matrix
      all_labels=y.to('cpu') #transfer to cpu (actual labels)
      all_preds=pred.argmax(1).to('cpu') #transfer to cpu(take the claass with the maximun likelihood for every image)
      acc=correct/total
      return avg_loss,acc,all_labels,all_preds



#Simple Convolutional Model

In [ ]:
class Net1(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1=nn.Conv2d(3,32,3)
    self.pool=nn.MaxPool2d(2,2)
    self.conv2=nn.Conv2d(32,64,3)
    self.conv3=nn.Conv2d(64,64,3)
    self.fc = nn.Linear(4*6*64, 7)


  def forward(self,x):
    x=self.pool(F.relu(self.conv1(x)))
    x=self.pool(F.relu(self.conv2(x)))
    x=self.pool(F.relu(self.conv3(x)))
    x=torch.flatten(x,1)
    x=self.fc(x)
    return x




In [ ]:
# Get cpu, gpu or mps device for training.
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

In [ ]:
model1=Net1()
model1.to(device)

In [ ]:
train_net(model1,trainloader,valloader,20,optim.SGD(model1.parameters(), lr=0.1),nn.CrossEntropyLoss().to(device),device,10)

In [ ]:
test_loss, test_accuracy,all_labels,all_preds= test_net(model1,testloader,nn.CrossEntropyLoss(),device)
print(test_loss,test_accuracy)
#Confusion matrix
cm=confusion_matrix(all_labels, all_preds)
disp=ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()


#Complicated Convolutional Model

In [ ]:
class Net2(nn.Module):
  def __init__(self):
    super().__init__()

    self.layers=nn.Sequential(
        nn.Conv2d(3,32,3),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(2,2),
        nn.Conv2d(32,64,3),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(2,2),
        nn.Conv2d(64,128,3),
        nn.ReLU(),
        nn.BatchNorm2d(128),
        nn.MaxPool2d(2,2),
        nn.Conv2d(128,256,3),
        nn.ReLU(),
        nn.BatchNorm2d(256),
        nn.MaxPool2d(2,2),
        nn.Conv2d(256,512,3),
        nn.ReLU(),
        nn.BatchNorm2d(512),
        nn.MaxPool2d(2,2)
    )
    self.linear=nn.Linear(512,7)


  def forward(self,x):
    x=self.layers(x)
    x=torch.nn.functional.adaptive_avg_pool2d(x,(1,1))
    x=torch.flatten(x,1)
    x=self.linear(x)
    return x


In [ ]:
model2=Net2()
model2.to(device)
#model.forward(full_dataset[10][0]) #64*130

In [ ]:
import torch.optim as optim

train_net(model2,trainloader,valloader,20,optim.Adam(model2.parameters(), lr=0.001),nn.CrossEntropyLoss(),device,10)

In [ ]:
test_loss, test_accuracy, all_labels,all_preds = test_net(model2,testloader,nn.CrossEntropyLoss(),device)
print(test_loss,test_accuracy)
#confusion matrix
cm_test = confusion_matrix(all_labels, all_preds)
disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test)
disp_test.plot()


#Transfer Learning

In [ ]:
data_transforms = {
  'train': transforms.Compose([
  transforms.RandomResizedCrop(224),
  transforms.RandomHorizontalFlip(),
  transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
  ]),

  'val': transforms.Compose([
  transforms.Resize(256),
  transforms.CenterCrop(224),
  transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
  ]),
}


In [ ]:
#apply the transformations to train and val_dataset
train_dataset.transform=data_transforms['train']
val_dataset.transform=data_transforms['val']

In [ ]:
resnet34=torchvision.models.resnet34(weights='DEFAULT')
resnet34.to(device)

In [ ]:
train_net(resnet34,trainloader,valloader,5,(optim.SGD(resnet34.parameters(), lr=0.001, momentum=0.9)),nn.CrossEntropyLoss(),device,10)

In [ ]:
test_loss, test_accuracy,all_preds,all_labels= test_net(resnet34,testloader,nn.CrossEntropyLoss(),device)
print(test_loss,test_accuracy)
cm=confusion_matrix(all_labels, all_preds)
disp=ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
